# 01 — Text Search with RelevanceIndex

`RelevanceIndex` is simlar's Text search engine. 

In this notebook we index 500 real Yelp reviews and run a few queries against them.

In [ ]:
%pip install -q datasets simlar

## Load the dataset

We use the [`yelp_review_full`](https://huggingface.co/datasets/yelp_review_full) dataset from HuggingFace — 650k restaurant and business reviews.

We take the first 500 from the test split to keep the demo fast. Each row has a `text` field (the review) and a `label` field (1–5 stars). We only need the text.

In [4]:
from datasets import load_dataset
from simlar import RelevanceIndex

ds = load_dataset("Yelp/yelp_review_full", split="test[:500]")
CORPUS = [text for text in ds["text"]]
IDS = [f"doc_{i}" for i in range(len(CORPUS))]  # simlar requires a unique string ID per document

print(f"Loaded {len(CORPUS)} reviews")
print(f"Sample: {CORPUS[0][:120]}")

Loaded 500 reviews
Sample: I got 'new' tires from them and within two weeks got a flat. I took my car to a local mechanic to see if i could get the


## Build the index

In [3]:
idx = RelevanceIndex()
idx.add(ids=IDS, texts=CORPUS)

# Keep a lookup dict so we can print the matched review text by ID
id_to_text = dict(zip(IDS, CORPUS))

print(f"Indexed {idx.size} documents")

TypeError: Argument 'texts' has incorrect type (expected list, got Column)

## Search

`search(query, k=3)` returns the top-k results as a list of `SearchResult` objects, each with:
- `rank` — position in the result list (1-based)
- `id` — the document ID you provided at index time
- `score` —  relevance score (higher = more relevant)


In [ ]:
# Positive query — looking for a specific dining experience
query = "cozy brunch spot great coffee"
results = idx.search(query, k=3)

print(f"Query: '{query}'\n")
for r in results:
    print(f"  rank={r.rank}  id={r.id}  score={r.score:.4f}")
    print(f"    {id_to_text[r.id][:120]}\n")

## Append a new document

In [ ]:
new_review = "amazing happy hour deals craft beer rooftop"

extended_corpus = list(CORPUS) + [new_review]
extended_ids    = list(IDS)    + ["doc_extra"]

idx.add(ids=extended_ids, texts=extended_corpus)  
id_to_text["doc_extra"] = new_review

print(f"After append: {idx.size} documents")

# The freshly added review should now rank #1 for this query
results3 = idx.search("craft beer happy hour", k=3)
for r in results3:
    print(f"  rank={r.rank}  id={r.id}  score={r.score:.4f}")
    print(f"    {id_to_text[r.id][:120]}\n")